# Serverless Data Pipeline with AWS Step Functions
> "Tutorial of Step Functions Python SDK to build serverless data pipelines"

- toc: true
- badges: true
- comments: true
- author: Anish Dalal
- categories: [AWS]

## Quick Summary

Data Scientists don't want to do infrastructure. Small teams and startups cannot manage complicated data infrastructure at the expense of shipping products. AWS Step Functions allow engineers and data scientists to deploy data pipelines without overhead costs such as managing continously running servers. Step functions orchestrate AWS services like ECS, Lambda, SageMaker, DynamoDB, etc. into complex workflows and DAGs. The Step Functions Python SDK simplifies development with Step Functions. This post explains how to use the SDK and its core abstractions to construct a toy example of a DAG all from a Jupyter notebook! Many of the concepts demonstrated in this tutorial can be used to create complex and production grade data/ML pipelines.

## Requirements

* AWS Account 
* [Install AWS CLI](https://docs.aws.amazon.com/cli/latest/userguide/inst)
* [Configure AWS CLI](https://docs.aws.amazon.com/cli/latest/userguide/cli-configure-quickstart.html)

This tutorial assumes some familiarity with boto3, AWS Step Functions, and AWS Lambda but is not necessary to run the notebook. Familiarity with [Amazon State Machine Language](https://states-language.net/spec.html) will also be helpful. 

In [ ]:
import sys
!{sys.executable} -m pip install --upgrade stepfunctions

## Create IAM Role and Policy for Executing Step Functions

In [1]:
import boto3
import json
iam_client = boto3.client('iam')

### Create IAM Role for this tutorial

We need to tell AWS that the step functions and lambda services can assume the IAM role we create.

In [2]:
trust_relationship_policy_doc = json.dumps({
  "Version": "2012-10-17",
  "Statement": [
      {
        "Effect": "Allow",
        "Principal": {"Service": "states.amazonaws.com"},
        "Action": "sts:AssumeRole"
      },
      {
        "Effect": "Allow",
        "Principal": {"Service": "lambda.amazonaws.com"},
        "Action": "sts:AssumeRole"
      }
  ]
})

role_name = 'StepFunctionsTutorialExecutionRole'

try:
    create_role_response = iam_client.create_role(
        RoleName=role_name,
        AssumeRolePolicyDocument=trust_relationship_policy_doc,
    )
except:
    print(f"{role_name} IAM Role has already been created")

### Create Lambda Execution Policy

In [3]:
lambda_execution_policy_doc = json.dumps({
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "lambda:InvokeFunction"
            ],
            "Resource": "*"
        }
    ]
})

policy_name = 'StepFunctionsTutorialExecutionPolicy'
try:
    create_policy_response = iam_client.create_policy(
        PolicyName=policy_name,
        PolicyDocument=lambda_execution_policy_doc,
    )
except:
    print(f"{policy_name} Policy has already been created")

### Attach Policy to Role

Attaching the policy doc to the IAM role gives any resource associated with IAM role permission to run a Lambda function.

In [4]:
tutorial_policy = list(filter(lambda x: x["PolicyName"] == policy_name, iam_client.list_policies()["Policies"]))
policy_arn = tutorial_policy[0]["Arn"]
attach_response = iam_client.attach_role_policy(RoleName=role_name, PolicyArn=policy_arn)

## Building a Toy Data Pipeline with Step Functions SDK

Our pipeline will consist of the following components:
1. Input -> 2 lists of 100 randomly generated numbers
2. Filtering Step -> Each list will be filtered
3. Averaging Step -> Each filtered list will be averaged
4. Decision Step -> Each average either undergoes a square root operation or is left alone

Each step is performed in a Lambda function.

Example:

input: [[0,5,-3...,174], [-2,5,7,...34]]

output: [-2,5.7]

In [5]:
# Helper Function to encode python file into byte string for creating Lambda functions
import zipfile
from io import BytesIO

lambda_client = boto3.client('lambda')
role_arn = iam_client.get_role(RoleName=role_name)["Role"]["Arn"]

def encode_file_to_bytes(file):
    buf = BytesIO()
    with zipfile.ZipFile(buf, 'w') as z: 
        z.write(file)
    buf.seek(0)
    return buf

### Create Filtering Step Lambda Function

In [6]:
%%writefile filter_step.py
def filter_numbers(event, context):
    items = event["items"]
    return [item for item in items if item < event["threshold"]]

Overwriting filter_step.py


In [7]:
filter_function_name = "filter-numbers"
try:
    filter_func_response = lambda_client.create_function(
        FunctionName=filter_function_name,
        Runtime='python3.8',
        Role=role_arn,
        Handler='filter_step.filter_numbers',
        Code={
            "ZipFile": encode_file_to_bytes("filter_step.py").read()
        }
    )
except:
    print(f"Lambda function {filter_function_name} has already been created")

Lambda function filter-numbers has already been created


### Create Averaging Step Lambda Function

In [8]:
%%writefile average_step.py
import statistics
def average_numbers(event, context):
    return statistics.mean(event["items"])

Overwriting average_step.py


In [9]:
average_function_name = "average-numbers"
try:
    average_func_response = lambda_client.create_function(
        FunctionName=average_function_name,
        Runtime='python3.8',
        Role=role_arn,
        Handler='average_step.average_numbers',
        Code={
            "ZipFile": encode_file_to_bytes("average_step.py").read()
        }
    )
except:
    print(f"Lambda function {average_function_name} has already been created")

### Create Decision Step Lambda Function

In [10]:
%%writefile decision_step.py
import math
def sqrt_number(event, context):
    return "output": math.sqrt(event["item"])

Overwriting decision_step.py


In [11]:
sqrt_function_name = "sqrt-number"
try:
    sqrt_response = lambda_client.create_function(
        FunctionName=sqrt_function_name,
        Runtime='python3.8',
        Role=role_arn,
        Handler='decision_step.sqrt_number',
        Code={
            "ZipFile": encode_file_to_bytes("decision_step.py").read()
        }
    )
except:
    print(f"Lambda function {sqrt_function_name} has already been created")

## Orchestrating Lambda Functions with Step Functions SDK

In [12]:
import stepfunctions
from stepfunctions.steps import *
from stepfunctions.workflow import Workflow

A Step Functions pipeline is a series of steps or "states" chained together. Within the toy example, each state is responsible for executing a specific Lambda function via a Lambda Step. Step Functions allows a developer to nest sub-pipelines within a state and make complicated workflows. 

Here are a couple of resources to help the reader understand the Amazon State Machine Language syntax used for defining inputs and outputs for each step of the pipeline.

* https://states-language.net/spec.html
* https://docs.aws.amazon.com/step-functions/latest/dg/concepts-input-output-filtering.html


![](step-functions-dag.png)

### Filter Step

This is the first step in our toy pipeline. It's using a [Parallel State](https://docs.aws.amazon.com/step-functions/latest/dg/amazon-states-language-parallel-state.html) so the branches will be executed simulatenously.

In [13]:
filter_step = Parallel(
    "Filter Lists of Numbers",
    input_path="$.list_of_number_lists",
    result_path="$"
)
filter_step.add_branch(
    LambdaStep(
        state_id="Exclude Numbers GTE 50 from List of Numbers",
        parameters={
            "FunctionName": filter_function_name,
            "Payload":{  
               "items.$":"$[0]",
               "threshold": 50
            }
        }
    )
)
filter_step.add_branch(
    LambdaStep(
        state_id="Exclude Numbers GTE 100 from List of Numbers",
        parameters={
            "FunctionName": filter_function_name,
            "Payload":{  
               "items.$":"$[1]",
               "threshold": 100
            }
        }
    )
)

### Average Step

In [14]:
average_step = LambdaStep(
    state_id="Average List of Numbers",
    input_path="$.item.Payload",
    result_path="$",
    parameters={
        "FunctionName": average_function_name,
        "Payload":{  
           "items.$":"$",
        }
    }
)

### Decision Step

Step Function support [Choice states](https://docs.aws.amazon.com/step-functions/latest/dg/amazon-states-language-choice-state.html) that can conditionally execute. In our example, if the resulting average is greater than 0 we take the square root otherwise usng the Pass State and leave the number alone.

In [15]:
decision_step = Choice(
    state_id="Is Average > 0?"
)

pass_step = Pass(
    state_id="Do Nothing"             
)

sqrt_step = LambdaStep(
    state_id="Square Root Number",
    result_path="$",
    parameters={
        "FunctionName": sqrt_function_name,
        "Payload":{  
           "item.$":"$.Payload.output",
        }
    }
)

decision_step.add_choice(
    rule=ChoiceRule.NumericLessThan(variable=average_step.output()["Payload"], value=0),
    next_step=pass_step
)
decision_step.add_choice(
    ChoiceRule.NumericGreaterThanEquals(variable=average_step.output()["Payload"], value=0),
    next_step=sqrt_step
)

### Chain average step and decision step

The average and decision steps are executed sequentially. Through the iterator parameter, the [Map state](https://docs.aws.amazon.com/step-functions/latest/dg/amazon-states-language-map-state.html) executes the average and decision steps sequentially for each list of numbers inputted to the Map step. The Map state is similar to the Parallel step because it executes its operations simulataneously but the key difference is the number of branches is dynamic and dependent on number of inputs specified in $$.Map.Item.Value.

In [16]:
average_and_decision_chain = Chain([average_step, decision_step])

map_step = Map(
    state_id="Average Step",
    result_path="$",
    parameters={
        "item.$": "$$.Map.Item.Value"
    },
    max_concurrency=2,
    iterator=average_and_decision_chain
)



succeess_step = Succeed("Pipeline Successful")

pipeline = Chain([filter_step, map_step, succeess_step])

workflow = Workflow(
    name="Pipeline",
    definition=pipeline,
    role=role_arn
)


workflow.render_graph()

### Execute Pipeline

In [17]:
if not workflow.list_workflows():
    workflow.create()

In [18]:
import random
workflow_execution = workflow.execute(inputs={
    "list_of_number_lists": [[random.randint(-200, 200) for _ in range(100)], [random.randint(-200, 200) for _ in range(100)]]
})

In [19]:
print([res["Payload"] for res in workflow_execution.get_output()])

[-73.61818181818182, -43.7037037037037]


## Clean Up

In [20]:
workflow.delete()

In [21]:
detach_response = iam_client.detach_role_policy(
    RoleName=role_name,
    PolicyArn=policy_arn
)
iam_client.delete_role(RoleName=role_name)
iam_client.delete_policy(PolicyArn=policy_arn)

{'ResponseMetadata': {'RequestId': '51033dd7-4bfb-43f2-bef0-224b33772dfc',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '51033dd7-4bfb-43f2-bef0-224b33772dfc',
   'content-type': 'text/xml',
   'content-length': '204',
   'date': 'Fri, 26 Jun 2020 23:25:45 GMT'},
  'RetryAttempts': 0}}

In [22]:
lambda_client.delete_function(FunctionName=filter_function_name)
lambda_client.delete_function(FunctionName=average_function_name)
lambda_client.delete_function(FunctionName=sqrt_function_name)

{'ResponseMetadata': {'RequestId': '4454384d-0af9-4609-aab1-098107549581',
  'HTTPStatusCode': 204,
  'HTTPHeaders': {'date': 'Fri, 26 Jun 2020 23:25:47 GMT',
   'content-type': 'application/json',
   'connection': 'keep-alive',
   'x-amzn-requestid': '4454384d-0af9-4609-aab1-098107549581'},
  'RetryAttempts': 0}}